# 🧹 02 - Tiền xử lý dữ liệu (Preprocessing)
**EduTalk HUIT — Hệ thống Tư vấn Ngành học**

Notebook này thực hiện:
- Xử lý missing values
- Loại bỏ outliers
- Encode nhãn (Label Encoding / One-Hot)
- Chuẩn hóa đặc trưng số (StandardScaler / MinMaxScaler)
- Xử lý mất cân bằng nhãn (SMOTE nếu cần)
- Chia tập train/val/test
- Lưu file đã xử lý vào `data/processed/`

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

OUTPUT_DIR = '../data/processed/'
MODEL_DIR  = '../models/'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(MODEL_DIR,  exist_ok=True)

print('✅ Libraries loaded')

## 1. Load dữ liệu thô

In [ ]:
df = pd.read_csv('../data/raw/huit_admissions_data.csv')
print(f'Raw shape: {df.shape}')
df.head()

## 2. Xử lý Missing Values

In [ ]:
print('Missing values trước xử lý:')
print(df.isnull().sum())

# TODO: Điều chỉnh chiến lược fill phù hợp với dữ liệu thật
# Cột số → fill bằng median
num_cols = df.select_dtypes(include='number').columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())

# Cột chuỗi → fill bằng mode
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print('\nMissing values sau xử lý:', df.isnull().sum().sum())

## 3. Loại bỏ Outliers (IQR)

In [ ]:
# TODO: Chỉnh score_cols theo tên cột thật
score_cols = ['toan', 'van', 'ly', 'hoa', 'sinh', 'anh', 'su', 'dia']
score_cols = [c for c in score_cols if c in df.columns]

before = len(df)
for col in score_cols:
    Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    IQR = Q3 - Q1
    df = df[(df[col] >= Q1 - 1.5*IQR) & (df[col] <= Q3 + 1.5*IQR)]

print(f'Xóa {before - len(df)} outlier rows. Còn lại: {len(df)}')

## 4. Encode nhãn (Label Encoding)

In [ ]:
# TODO: Thay 'major' bằng tên cột nhãn thật
LABEL_COL = 'major'

le = LabelEncoder()
df['label'] = le.fit_transform(df[LABEL_COL])

# Lưu label encoder để dùng khi deploy
joblib.dump(le, MODEL_DIR + 'label_encoder.pkl')
print(f'✅ Label encoder saved. Classes: {le.classes_}')

# Mapping
label_map = dict(zip(le.classes_, le.transform(le.classes_)))
print('\nLabel mapping:')
for k, v in label_map.items():
    print(f'  {v:2d}: {k}')

## 5. Chuẩn hóa đặc trưng số

In [ ]:
# TODO: Cập nhật feature_cols
feature_cols = score_cols  # Hoặc thêm các cột đặc trưng khác

scaler = StandardScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])

# Lưu scaler
joblib.dump(scaler, MODEL_DIR + 'scaler.pkl')
print('✅ Scaler saved')
df[feature_cols].describe()

## 6. Chia tập Train / Validation / Test (70/15/15)

In [ ]:
X = df[feature_cols]
y = df['label']

# Split train vs temp (85% / 15%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
# Split val vs test (50% / 50% của temp)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f'Train:      {X_train.shape[0]:,} samples ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Validation: {X_val.shape[0]:,} samples ({X_val.shape[0]/len(X)*100:.0f}%)')
print(f'Test:       {X_test.shape[0]:,} samples ({X_test.shape[0]/len(X)*100:.0f}%)')

## 7. Lưu dữ liệu đã xử lý

In [ ]:
X_train.to_csv(OUTPUT_DIR + 'X_train.csv', index=False)
X_val.to_csv(OUTPUT_DIR   + 'X_val.csv',   index=False)
X_test.to_csv(OUTPUT_DIR  + 'X_test.csv',  index=False)
y_train.to_csv(OUTPUT_DIR + 'y_train.csv', index=False)
y_val.to_csv(OUTPUT_DIR   + 'y_val.csv',   index=False)
y_test.to_csv(OUTPUT_DIR  + 'y_test.csv',  index=False)

print('✅ Đã lưu tất cả file vào data/processed/')
print('→ Chạy tiếp: 03_model_training.ipynb')